<a href="https://colab.research.google.com/github/melissa-04/spink1-gbm-hypoxia/blob/main/01_cohort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 Discovery cohort

Builds the TCGA PanCancer Atlas IDH-wildtype glioblastoma cohort and the filtered expression matrix used in all TCGA analyses. Produces Table S1 and Table S2.

In [1]:
%%capture
!pip install -q gseapy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, tarfile
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE = '/content/drive/MyDrive/SPINK1'
REPO = os.path.join(BASE, 'repo')
RESULTS = os.path.join(REPO, 'results')
PROCESSED = os.path.join(REPO, 'processed')
sys.path.append(REPO)
from spink1_utils import spink1_groups, GROUP_ORDER

with tarfile.open(os.path.join(BASE, 'gbm_tcga_pan_can_atlas_2018.tar.gz')) as t:
        t.extractall('/content/tcga', filter='data')
STUDY = os.path.dirname(glob.glob('/content/tcga/*/data_clinical_patient.txt')[0])
print(STUDY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/tcga/gbm_tcga_pan_can_atlas_2018


In [3]:
expr = pd.read_csv(os.path.join(STUDY, 'data_mrna_seq_v2_rsem.txt'), sep='\t', low_memory=False)
expr = expr[expr['Hugo_Symbol'].notna()].copy()
samples = [c for c in expr.columns if c.startswith('TCGA')]
expr[samples] = expr[samples].apply(pd.to_numeric, errors='coerce')
expr = expr.copy()

expr['mean_expr'] = expr[samples].mean(axis=1)
expr = expr.sort_values('mean_expr', ascending=False).drop_duplicates('Hugo_Symbol')
expr = expr.set_index('Hugo_Symbol')[samples]

L = np.log2(expr.clip(lower=0) + 1)
expressed = (L > 1).sum(axis=1) >= 0.2 * len(samples)

print('Samples:', len(samples))
print('Genes:', L.shape[0], '| expressed:', int(expressed.sum()))

Samples: 160
Genes: 20511 | expressed: 17754


In [4]:
patient = pd.read_csv(os.path.join(STUDY, 'data_clinical_patient.txt'), sep='\t', comment='#')
hypoxia = pd.read_csv(os.path.join(STUDY, 'data_clinical_supp_hypoxia.txt'), sep='\t', comment='#')
mut = pd.read_csv(os.path.join(STUDY, 'data_mutations.txt'), sep='\t', comment='#', low_memory=False)

sequenced = set(mut['Tumor_Sample_Barcode'])
idh_mutant = set(mut.loc[mut['Hugo_Symbol'].isin(['IDH1', 'IDH2']), 'Tumor_Sample_Barcode'])

primary = sorted(s for s in samples if s.split('-')[3] == '01')
cohort_ids, seen = [], set()
for s in primary:
    if s[:12] not in seen:
        seen.add(s[:12])
        cohort_ids.append(s)

coh = pd.DataFrame({'SAMPLE_ID': cohort_ids})
coh['PATIENT_ID'] = coh['SAMPLE_ID'].str[:12]
coh['IDH_status'] = np.select([coh['SAMPLE_ID'].isin(idh_mutant), coh['SAMPLE_ID'].isin(sequenced)],
                              ['Mutant', 'WT'], default='Not available')
coh['SPINK1_log2'] = L.loc['SPINK1', cohort_ids].values
coh = coh.merge(patient[['PATIENT_ID', 'AGE', 'SEX', 'SUBTYPE', 'OS_STATUS', 'OS_MONTHS']],
                on='PATIENT_ID', how='left').merge(hypoxia, on='PATIENT_ID', how='left')

excluded = (coh['IDH_status'] == 'Mutant') | (coh['SUBTYPE'] == 'GBM_IDHmut-non-codel')
coh = coh[~excluded].reset_index(drop=True)
coh['SPINK1_group'] = spink1_groups(coh['SPINK1_log2']).values

print('Primary tumors, one per patient:', len(cohort_ids))
print('Excluded as IDH-mutant:', int(excluded.sum()))
print('IDH-wildtype cohort:', len(coh))
print(coh['IDH_status'].value_counts().to_string())
print(coh['SPINK1_group'].value_counts().reindex(GROUP_ORDER).to_string())

Primary tumors, one per patient: 153
Excluded as IDH-mutant: 9
IDH-wildtype cohort: 144
IDH_status
WT               141
Not available      3
SPINK1_group
Undetectable    56
Low             44
High            44


In [5]:
coh.to_csv(os.path.join(RESULTS, 'Table_S1_cohort.csv'), index=False)
L.loc[expressed, coh['SAMPLE_ID']].to_parquet(os.path.join(PROCESSED, 'tcga_expression.parquet'))

print(pd.read_csv(os.path.join(RESULTS, 'Table_S1_cohort.csv')).shape)
print(pd.read_parquet(os.path.join(PROCESSED, 'tcga_expression.parquet')).shape)

(144, 13)
(17754, 144)


In [6]:
cna = pd.read_csv(os.path.join(STUDY, 'data_cna.txt'), sep='\t', low_memory=False)
row = cna[cna['Hugo_Symbol'] == 'SPINK1'].iloc[0]
ids = [s for s in coh['SAMPLE_ID'] if s in cna.columns]

d = pd.DataFrame({'cna': pd.to_numeric(row[ids], errors='coerce'),
                  'expr': coh.set_index('SAMPLE_ID').loc[ids, 'SPINK1_log2']}).dropna()
r, p = spearmanr(d['cna'], d['expr'])

s2 = pd.DataFrame([{'driver': 'Copy number (GISTIC)', 'region': 'gene-level',
                    'n': len(d), 'rho': round(r, 3), 'p': round(p, 3)}])
s2.to_csv(os.path.join(RESULTS, 'Table_S2_SPINK1_copy_number.csv'), index=False)
print(s2.to_string(index=False))

              driver     region   n   rho     p
Copy number (GISTIC) gene-level 140 0.055 0.522
